In [ ]:
from config_dirs import PERPLEXITY_INPUT, STANDARDIZED_GENES_OUTPUT

import sys
print(sys.version)
import openai
print(openai.__version__)
import pandas as pd
import requests
import os
from openai import OpenAI

3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]


In [ ]:
pd.read_csv(PERPLEXITY_INPUT).query("`Entity of Association` == 'Protein'").to_excel('OpenAI_Output_Protein_only.xlsx', index=False)

file_name = 'OpenAI_Output_Protein_only.xlsx'
import pandas as pd
# Load the Excel file into a pandas DataFrame
df = pd.read_excel(file_name)



,PMID,Entity of Association,Name,Variability Type,Causality,Significance,Disease/Pathology,Organism,Organ/System,Sample Size,Population Ethnicity,Abstract Reference,Abstract Reference
0,31630160,Protein,CCDC93,Increased protein stability,Causal,"Functional assays (overexpression in mice, abl...",Lower LDL-c levels,"Human, Mouse",Cardiovascular system,"N=107,000 (Copenhagen studies); validated in m...",Not specified,"""The variant is shown to increase CCDC93 prote...",NaN
1,34234248,Protein,PLXNA4,Decreased levels,Consequential,Association with worsened respiratory function...,"Pulmonary embolism (PE), Acute lung injury",Human,"Respiratory system (lungs), Cardiovascular sys...",N=112 (COVID-19 patients),Not specified,NaN,"""In a sample of 112 COVID-19 patients known to..."
2,36411224,Protein,Myeloperoxidase (MPO),Increased plasma levels,Causal,"OR=1.05 (95% CI 1.02â€“1.09, P=0.002) for isch...","Ischemic stroke, Cardioembolic stroke (CES), H...",Human,Cardiovascular system,"N=440,328 (European individuals)",European,"""Genetically determined higher plasma MPO conc...",NaN
3,36696485,Protein,ALDH2,Severe loss of enzymatic activity,Causal,"Functional assays (iPSC-ECs, CRISPR-Cas9 corre...","Coronary artery disease (CAD), Endothelial cel...",Human,Cardiovascular system,Not specified,Not specified,"""Even in heterozygous carriers, this missense ...",NaN
4,36696485,Protein,Nitric oxide (NO),Decreased production,Consequential,"Functional assays (iPSC-ECs, CRISPR-Cas9 corre...",Endothelial cell dysfunction,Human,Cardiovascular system,Not specified,Not specified,"""Using human induced pluripotent stem cell-der...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,34969212,Protein,"Ion channels, receptors, scaffolding proteins,...",Structural changes due to gene mutations,Consequential,Reported in clinical syndromes,Pain syndromes (general),Human,"Nervous system (neurons, spinal tracts)",Not specified,Not specified,"""Structural changes of proteins caused by gene...",NaN
222,36113314,Protein,Complement factor H (FH),Increased serum levels,Consequential,p=0.004,Immunoglobulin A nephropathy (IgAN) with micro...,Human,Renal system (Kidney),"N=2055 (204 with MA, 1851 without MA)",Not specified,"""Patients with MA lesions are strongly associa...",NaN
223,36300369,Protein,CNNM2,Higher levels associated with increased risk,Causal,P=3.39Ã—10^-16; posterior probability >70%; id...,Intracranial aneurysms (IA),Human,Cardiovascular system (arterial tissue),Not specified,Not specified,"""Higher CNNM2 levels in arterial tissue were a...",NaN
224,36737746,Protein,GLP-1 receptor (GLP1R),Increased expression (genetically-predicted),Causal,"OR=0.83 (95% CI, 0.71-0.97) for background DR;...","Diabetic retinopathy (background DR, severe no...",Human,Visual system (Eye),N=2390 (observational study); FinnGen cohort (...,Not specified,"""Genetically-predicted GLP1R expression (the t...",NaN


In [ ]:
import time
import pandas as pd
import openai
from openai import OpenAI
# Initialize OpenAI client
YOUR_API_KEY = "your_api_key_here"
client = openai.OpenAI(api_key=YOUR_API_KEY, base_url="https://api.perplexity.ai")

# Function to query the LLM for the standardized gene/protein name
def get_standardized_name(gene_name, cache):
    if gene_name in cache:
        return cache[gene_name]
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI assistant with expert-level knowledge in genomics and molecular biology. "
                "Please return only the Gene Symbol based on the HUGO Gene Nomenclature Committee (HGNC) guidelines without any additional text or explanation."
            ),
        },
        {
            "role": "user",
            "content": f"{gene_name}",
        },
    ]

    # Perform the API call
    response = client.chat.completions.create(
        model="llama-3.1-sonar-huge-128k-online",
        messages=messages,
    )
    
    # Extract and return the standardized name from the response
    standardized_name = response.choices[0].message.content
    
    # Cache the result
    cache[gene_name] = standardized_name
    
    return standardized_name

# Function to process the DataFrame with rate limiting, caching, and retry logic
def process_dataframe(df, save_interval=20, max_retries=3):
    request_count = 0
    processed_count = 0
    cache = {}
    
    for index, row in df.iterrows():
        gene_name = row['Name']
        
        for retry in range(max_retries):
            try:
                standardized_name = get_standardized_name(gene_name, cache)
                df.at[index, 'Perplexity'] = standardized_name
                print(f"Processed row {index}: {gene_name} -> {standardized_name}")
                
                # Increment the request count only if it's not retrieved from cache
                if gene_name not in cache:
                    request_count += 1
                
                processed_count += 1
                
                # Check if rate limit is close to being exceeded
                if request_count >= 20:
                    # Save progress to a file
                    df.to_csv('standardized_genes_partial.csv', index=False)
                    print(f"Reached rate limit, saved progress after processing {processed_count} rows.")
                    
                    # Reset request count and sleep for 60 seconds
                    request_count = 0
                    time.sleep(60)
                
                break  # Exit retry loop if successful
            except Exception as e:
                if "rate limit" in str(e).lower():
                    if retry < max_retries - 1:
                        print(f"Rate limit exceeded. Retrying in 60 seconds...")
                        time.sleep(60)
                    else:
                        raise
                else:
                    print(f"Error processing row {index}: {e}")
                    break  # Exit retry loop if there's an error other than rate limit
        
        # Save progress periodically
        if processed_count % save_interval == 0:
            df.to_csv('standardized_genes_partial.csv', index=False)
            print(f"Saved progress after processing {processed_count} rows.")
    
    # Final save after all rows are processed
    df.to_csv(STANDARDIZED_GENES_OUTPUT, index=False)
    print("Finished processing all rows.")

# Example setup for the DataFrame (this should be your actual DataFrame)
# df = pd.read_csv('your_data.csv')  # Load your actual data here

# Initialize a 'Perplexity' column in the DataFrame
df['Perplexity'] = None



In [6]:
process_dataframe(df)

Processed row 0: CCDC93 -> CCDC93
Processed row 1: PLXNA4 -> PLXNA4
Processed row 2: Myeloperoxidase (MPO) -> MPO
Processed row 3: ALDH2 -> ALDH2
Processed row 4: Nitric oxide (NO) -> NOS1, NOS2, NOS3[1][3]
Processed row 5: Na+/H+-exchanger 1 (NHE-1) -> SLC9A1
Processed row 6: AKT kinase -> AKT1, AKT2, AKT3[1][2][5]
Processed row 7: Endothelial NO synthase (eNOS) -> NOS3
Processed row 8: CETP (Cholesteryl Ester Transfer Protein) -> CETP
Processed row 9: RANTES (CCL5) -> CCL5
Processed row 10: EOTAXIN (CCL11) -> CCL11
Processed row 11: Interleukin 18 (IL-18) -> IL18
Processed row 12: PILRA -> PILRA
Processed row 13: GRN -> GRN
Processed row 14: APOL3 -> APOL3
Processed row 15: LRP4 -> LRP4
Processed row 16: F11 -> F11 (Coagulation factor XI)
Processed row 17: Hepatic lipase (HL) -> LIPC
Processed row 18: Small dense LDL (sdLDL) -> No specific gene symbol is directly associated with "small dense LDL (sdLDL)" as it refers to a subtype of low-density lipoprotein cholesterol rather than a g

Processed row 98: TNF-related apoptosis-inducing ligand (TRAIL) -> TNFSF10
Processed row 99: Interleukin-1 receptor-like 2 (IL-1RL2) -> IL1RL2
Saved progress after processing 100 rows.
Processed row 100: Interleukin-18 (IL-18) -> IL18
Processed row 101: E-selectin -> SELE
Processed row 102: IL-3 receptor subunit alpha (IL-3Ra) -> IL3RA
Processed row 103: IL-5 receptor subunit alpha (IL-5Ra) -> IL5RA
Processed row 104: SMOC1 -> SMOC1
Processed row 105: TIE1 -> TIE1
Processed row 106: NDUFB4 -> NDUFB4
Processed row 107: ETHE1 -> ETHE1
Processed row 108: POFUT2 -> POFUT2
Processed row 109: TRIL -> TRIL1
Processed row 110: ADAM23 -> ADAM23
Processed row 111: GXYLT1 -> GXYLT1
Processed row 112: OXT -> OXT is not a gene symbol recognized by the HUGO Gene Nomenclature Committee (HGNC). The term "OXT" in the provided search results refers to a cryptocurrency and a musical group, not a gene.

However, "OXT" could be confused with "OXT" as an abbreviation for oxytocin, which is a hormone. The co

Processed row 190: IgD-CD38dim B cell -> No specific gene symbol is directly associated with the term "IgD-CD38dim B cell" as it describes a phenotype rather than a specific gene. However, genes relevant to the markers mentioned are:

- **IGD** for IgD
- **CD38** for CD38

These are not gene symbols for a specific B cell subset but rather for the markers used to identify such subsets.
Processed row 191: Unswitched memory B cell -> No specific gene symbol is directly associated with "unswitched memory B cells" as they are a subset of B cells characterized by their expression of CD27 and IgD (CD27+IgD+), rather than a specific gene. However, genes mentioned in the context of unswitched memory B cells include:

- **BCL6** (involved in germinal center reactions and mutated in some unswitched memory B cells)[1]
- **IGHV** (involved in BCR repertoire and biased usage in autoimmune diseases)[1]
- **CD27** (marker for memory B cells)[1][2][3]
- **CD21** (lost in activated unswitched memory B c